# LLM Evaluation on Colab T4

Ноутбук для проверки качества `RAG + LLM` на вашей базе нормативки.

Загрузите в Colab файлы:
- `processed/lexical_index.json`
- `data/test/eval_questions.jsonl`


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

In [ ]:
# Надежный вариант без JS-виджета files.upload()
# 1) Скопируйте файлы в Google Drive, например:
#    /content/drive/MyDrive/egais_eval/lexical_index.json
#    /content/drive/MyDrive/egais_eval/eval_questions.jsonl
# 2) Укажите путь ниже

from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/egais_eval')  # <- при необходимости поменяйте

index_src = DRIVE_DIR / 'lexical_index.json'
questions_src = DRIVE_DIR / 'eval_questions.jsonl'

assert index_src.exists(), f'Файл не найден: {index_src}'
assert questions_src.exists(), f'Файл не найден: {questions_src}'

shutil.copy(index_src, '/content/lexical_index.json')
shutil.copy(questions_src, '/content/eval_questions.jsonl')

print('Ready files in /content:', '/content/lexical_index.json', '/content/eval_questions.jsonl')

In [ ]:
import json, math, re
from collections import Counter

TOKEN_RE = re.compile(r"[a-zA-Zа-яА-Я0-9]{2,}")

def tokenize(text):
    return [t.lower() for t in TOKEN_RE.findall(text)]

def doc_weight(row):
    meta = row.get('metadata', {})
    source = (meta.get('source_file') or '').lower()
    doc_type = (meta.get('doc_type') or '').upper()
    is_official = doc_type in {'ПРИКАЗ','ПОСТАНОВЛЕНИЕ','РАСПОРЯЖЕНИЕ','ФЕДЕРАЛЬНЫЙ ЗАКОН'}
    if not is_official:
        return 0.0
    weight = 1.25
    if source.startswith('guide_'):
        weight *= 0.75
    if 'unknown' in source:
        weight *= 0.65
    return weight

def score_query(query, index):
    q_tf = Counter(tokenize(query))
    if not q_tf:
        return []
    idf = index['idf']
    scored = []
    for d in index['docs']:
        w = doc_weight(d)
        if w <= 0:
            continue
        score = 0.0
        d_tf = d['tf']
        d_len = max(1, d['len'])
        for tok, qf in q_tf.items():
            if tok in d_tf and tok in idf:
                score += (qf * idf[tok]) * (d_tf[tok] * idf[tok] / math.sqrt(d_len))
        if score > 0:
            scored.append((score * w, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored

with open('lexical_index.json', 'r', encoding='utf-8') as f:
    index = json.load(f)

questions = []
with open('eval_questions.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            questions.append(json.loads(line))

print('Questions:', len(questions), '| Indexed docs:', index['n_docs'])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    load_in_4bit=True,
)
print('Model loaded')

In [ ]:
def build_prompt(question, matches):
    blocks = []
    for i, (score, row) in enumerate(matches, 1):
        m = row.get('metadata', {})
        doc_type = m.get('doc_type', 'Документ')
        doc_no = m.get('doc_number_file') or m.get('doc_number_text') or 'n/a'
        source = m.get('source_file', 'n/a')
        blocks.append(f"[{i}] {doc_type} №{doc_no} ({source})\n{row['text'][:1200]}")
    context = '\n\n'.join(blocks)
    return (
        'Ты юридический ассистент по лицензированию ЕГАИС.\n'
        'Отвечай только по предоставленному контексту.\n'
        'Если данных недостаточно, скажи об этом.\n'
        'Не выдумывай нормативные реквизиты.\n'
        'Формат: 1) Краткий ответ 2) Нормативное основание 3) Практические шаги 4) Источники\n\n'
        f'Вопрос:\n{question}\n\nКонтекст:\n{context}'
    )

def llm_answer(question, top_k=6, max_new_tokens=500):
    matches = score_query(question, index)[:top_k]
    prompt = build_prompt(question, matches)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = text[len(prompt):].strip() if text.startswith(prompt) else text
    return answer, matches

In [ ]:
results = []
for q in questions:
    answer, matches = llm_answer(q['question'], top_k=6)
    results.append({
        'id': q['id'],
        'question': q['question'],
        'retrieved_sources': [m[1].get('metadata', {}).get('source_file', 'n/a') for m in matches],
        'answer': answer,
    })
    print('Done:', q['id'])

with open('eval_colab_t4.jsonl', 'w', encoding='utf-8') as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Saved eval_colab_t4.jsonl')

In [ ]:
# Сохраняем результат в Google Drive (без JS-виджета download)
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
OUT_DIR = Path('/content/drive/MyDrive/egais_eval')
OUT_DIR.mkdir(parents=True, exist_ok=True)

src = Path('/content/eval_colab_t4.jsonl')
dst = OUT_DIR / 'eval_colab_t4.jsonl'
shutil.copy(src, dst)
print('Saved to:', dst)